## Training and Evaluation

In [1]:
%pip install nbformat


   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%capture
%run Preprocessing.ipynb

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold,GridSearchCV
import numpy as np
from sklearn.metrics import  accuracy_score,f1_score, balanced_accuracy_score, classification_report
x_tr_clean = x_tr_clean
y_tr = y_tr
y_trv = y_trv
y_v = y_v
x_v_clean = x_v_clean#reusing the data from the previous notebook
x_te_clean = x_te_clean
y_te = y_te
num_cols = num_cols
total_pipe = total_pipe
x_trv = x_trv

In [22]:
pca = PCA(n_components=0.9)
x_pca_tr = pca.fit_transform(x_tr_clean) 
x_pca_v = pca.transform(x_v_clean)

# Now our data is scaled and with reduced dimensions
def majority(y_tr,x_v):
   majority = np.unique_counts(y_tr).values[np.argmax(np.unique_counts(y_tr).counts)]
   pred = np.ones(len(x_v))
   return majority*pred
print(f"Majority class macro f1 score is {f1_score(y_v, majority(y_tr,x_pca_v),average="macro"):.2f}")

LR_model = LogisticRegression(random_state=10)
LR_model.fit(x_pca_tr,y_tr)
print(f"Multinomial Logistic Regression macro f1 score is  {f1_score(y_v, LR_model.predict(x_pca_v),average="macro"):.2f} ")
Tree_model = DecisionTreeClassifier(random_state=10)
Tree_model.fit(x_pca_tr,y_tr)
print(f"Decision Tree macro f1 score is  {f1_score(y_v, Tree_model.predict(x_pca_v),average="macro"):.2f} ")
RFC_model = RandomForestClassifier(random_state=10)
RFC_model.fit(x_pca_tr,y_tr)
print(f"Random Forest macro f1 score is  {f1_score(y_v, RFC_model.predict(x_pca_v),average="macro"):.2f} ")

Majority class macro f1 score is 0.22
Multinomial Logistic Regression macro f1 score is  0.86 
Decision Tree macro f1 score is  0.72 
Random Forest macro f1 score is  0.84 


In [44]:
kfold = StratifiedKFold(n_splits=5, random_state=10, shuffle=True)
def param_tuning(model,param_grid,name):
    pipe = Pipeline([("preprocess",total_pipe),("pca",pca),("model",model)])
    grid = GridSearchCV(pipe,param_grid,cv = kfold,scoring="f1_macro")
    grid.fit(x_trv,y_trv) #using x_trv since validation will automatically split train and valid
    mean_f1 = grid.cv_results_["mean_test_score"][grid.best_index_]
    std_f1 = grid.cv_results_["std_test_score"][grid.best_index_]
    param = list(grid.best_params_.keys())[0].replace("model__","")
    value = list(grid.best_params_.values())[0]
    print(f"For {name} best parameter is {param} = {value}\nThis gives mean macro F1 score of {mean_f1:.3f} with deviation of {std_f1:.3f}")
param_tuning(LogisticRegression(random_state=10),{"model__C": [0.01, 0.1, 1.0, 10.0]} ,"Logistic Regression")
param_tuning(DecisionTreeClassifier(random_state=10),{"model__max_depth": [1, 3, 7, None]} ,"Decision Tree")
param_tuning(RandomForestClassifier(random_state=10),{"model__n_estimators": [50, 100, 200]},"Random Forest" )



For Logistic Regression best parameter is C = 1.0
This gives mean macro F1 score of 0.801 with deviation of 0.070
For Decision Tree best parameter is max_depth = 3
This gives mean macro F1 score of 0.708 with deviation of 0.128
For Random Forest best parameter is n_estimators = 200
This gives mean macro F1 score of 0.759 with deviation of 0.049


In [45]:
def evaluate(model, x_te_clean, y_te):
    model.fit(x_pca_tr,y_tr)
    y_pred = model.predict(pca.transform(x_te_clean))

    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    F1 = f1_score(y_te, y_pred, average="macro")
    per_class_prec = classification_report(y_te, y_pred)

    return acc, bal_acc, F1, per_class_prec
LR_model = LogisticRegression( random_state=10,C=1)
Tree_model=DecisionTreeClassifier(random_state=10, max_depth=3)
RFC_model = RandomForestClassifier(random_state=10,n_estimators=200)
evaluate(LR_model, x_te_clean, y_te)
evaluate(Tree_model, x_te_clean, y_te)
evaluate(RFC_model, x_te_clean, y_te)

(0.9487179487179487,
 0.9111111111111111,
 0.934802043422733,
 '              precision    recall  f1-score   support\n\n         0.0       1.00      0.80      0.89         5\n         1.0       0.90      1.00      0.95        19\n         2.0       1.00      0.93      0.97        15\n\n    accuracy                           0.95        39\n   macro avg       0.97      0.91      0.93        39\nweighted avg       0.95      0.95      0.95        39\n')

In [ ]:
x_num_clean = x_tr_clean[num_cols] # no data leakage between num and categorical columns
cat_cols_new = x_tr_clean.columns[~np.isin(x_tr_clean.columns,num_cols)] #different from original cat_cols due to OHE
x_cat_clean = x_tr_clean[cat_cols_new] # missing indicator columns count towards categorical features
x_num_pca = pca.fit_transform(x_num_clean)
LR_model.fit(x_num_pca,y_tr)
x_num_v = pca.transform(x_v_clean[num_cols])
print(f"Numerical only LR F1 score is {f1_score(y_v,LR_model.predict(x_num_v),average="macro")}")

pca_cat = PCA(n_components=0.9) 
x_cat_pca = pca_cat.fit_transform(x_cat_clean)
LR_model.fit(x_cat_pca,y_tr)
x_cat_v = pca_cat.transform(x_v_clean[cat_cols_new])
print(f"Categorical only LR F1 score is {f1_score(y_v,LR_model.predict(x_cat_v),average="macro")}")

x_combined_pca = pca.fit_transform(x_tr_clean)
LR_model.fit(x_combined_pca,y_tr)
x_combined_v = pca.transform(x_v_clean)
print(f"Combined LR F1 score is {f1_score(y_v,LR_model.predict(x_combined_v),average="macro")}")

Numerical only LR F1 score is 0.8107031857031858
Categorical only LR F1 score is 0.6352657004830918
Combined LR F1 score is 0.860377973281199
